In [1]:
from dotenv import load_dotenv  # .env 환경변수 로드
from openai import OpenAI  # 클라이언트 객체
import os  # 환경변수 접근

load_dotenv()  # .env파일 읽어와 환경변수 등록
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # .env파일의 OPENAI_API_KEY 값 가져옴
client = OpenAI()  # OPENAI_API_KEY가 환경변수에 등록되어 있으면 (api_key = ...) 생략 가능

In [ ]:
# Chat Completion API 호출
response = client.chat.completions.create(
    model = 'gpt-5.6-luna',
    messages = [
        # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
        {
            "role": "system",
            "content": '당신은 세상에서 가장 친절한 챗봇입니다.'
        },
        # 유저 프롬프트 : 사용자의 실제 입력 데이터
        {
            "role": "user",  # 사용자 대화 1
            "content": '안녕, 나는 LLM을 공부하는 학생이야~ 만나서 반갑다!'
        },
        {
            "role": "assistant",  # (이전 대화 내용 중) 챗봇의 답변 1
            "content": '안녕! 만나서 반가워~ LLM 공부를 하고 있구나. 어떤 분야에 관심이 있는지 말해줄래?'
        },
        {
            "role": "user",  # 사용자 대화 2
            "content": 'GPT 모델은 트랜스포머와 어떤 연관성을 가지고 있는거야?'
        },
        {
            "role": "assistant",  # (이전 대화 내용 중) 챗봇의 답변 2
            "content": """
좋은 질문이야! GPT 모델은 트랜스포머(Transformer) 아키텍처를 기반으로 발전했어. 간단히 설명할게.

1. **트랜스포머(Transformer)**
2017년에 "Attention is All You Need" 논문에서 소개된 트랜스포머는 자연어 처리의 혁신적인 모델 구조야. 이전 RNN이나 LSTM과 달리, 순차적인 계산 없이도 병렬 처리가 가능하고, '셀프 어텐션(Self-Attention)' 메커니즘을 사용해 문맥을 효율적으로 이해할 수 있어.

2. **GPT (Generative Pre-trained Transformer)**
GPT는 트랜스포머의 '디코더' 부분을 기반으로 한 모델이야. OpenAI가 처음 발표한 GPT는 대규모 텍스트 데이터에 대해 '언어 모델링' 방식으로 사전학습(pre-training)을 하고, 이후 특정 작업에 맞게 미세조정(fine-tuning)하는 방식을 사용했어.

3. **주요 발전 포인트**
- **사전학습과 미세조정**: GPT는 트랜스포머를 이용해 방대한 텍스트를 예측하는 작업으로 사전학습하고, 이후 다양한 언어 처리 작업에 적용할 수 있어.
- **규모 확장**: GPT-2, GPT-3, 그리고 최신 GPT-4로 갈수록 모델 크기와 학습 데이터 양이 크게 늘어나면서, 더 자연스럽고 다재다능한 언어 능력을 갖추게 됐어.
- **제로샷/원샷 학습 능력**: 큰 모델로 발전하면서 GPT는 별도 미세조정 없이도 (제로샷) 또는 적은 예시만으로도 (원샷/퓨샷) 다양한 작업을 수행할 수 있게 되었어.

정리하자면, GPT는 트랜스포머 아키텍처의 강력한 특징들을 활용해, 대규모 사전학습과 점차 확장된 모델 크기를 통해 자연어 생성 및 이해 능력을 크게 향상시킨 모델이라고 할 수 있어.

더 궁금한 점 있으면 언제든 물어봐!
            """
        },
        {
            "role": "user",  # 사용자 대화 3
            "content": '와 너무 도움이 많이 됐어 고마워. 근데 세줄 요약좀!'
        },
    ],
    temperature = 1,               # 창의성/다양성 (낮으면 결정론적, 일관적 / 높으면 창의적)
    max_completion_tokens = 4096,  # 최대 출력 토큰
    top_p = 1                      # 누적확률 p까지의 후보를 샘플링 (1은 전체사용)
)

response

In [ ]:
print(response.choices[0].message.content)

In [ ]:
messages = [{'role': 'system', 'content': '당신은 지혜로운 챗봇입니다.'}]

print("종료하려면 exit를 입력하세요.")

while True:
    user_input = input('User : ')

    if user_input.lower().strip() == 'exit':
        print("대화 종료됩니다.")
        break

    print(f"User : {user_input}")
    messages.append({'role': 'user', 'content': user_input})

    response = client.chat.completions.create(
        model = 'gpt-5.6-luna',
        messages = messages,
        temperature = 1,
        top_p = 1,
        max_completion_tokens = 4096
    )
    assistant_message = response.choices[0].message.content
    print(f"Assistant : {assistant_message}")
    messages.append({'role': 'assistant', 'content': assistant_message})

- 일반 서비스에서 대화 저장을 처리하는 방식
    
1) DB에 저장 (가장 정석)
    
    - **RDB(PostgreSQL/MySQL)**: 유저/세션/메시지 관계가 명확해서 운영에 강함
    - **NoSQL(MongoDB 등)**: 문서 형태로 대화 전체를 한 번에 저장하기 편함
    
2) 캐시에 저장 (속도용) + DB에 영구 저장(혼합)
    
    - **Redis**에 최근 N개 메시지를 캐시로 두고 빠르게 응답
    - 일정 주기/이벤트 기준으로 **DB에 영구 저장**
    - 장점: 빠르고 비용 효율적, 단점: Redis만 쓰면 영속성 약함(그래서 DB와 같이 씀)
    
3) 파일로 저장 (개인/PoC에서 가장 간단)
    
    - JSON/JSONL 파일로 로컬에 저장
    - 운영 서비스에서는 보통 DB로 가지만, 학습/실습/로컬 챗봇엔 매우 유용

- 서비스에서 꼭 같이 고려하는 포인트
    
A) “전체 대화”를 매번 모델에 다 보내면 비용/속도 문제가 생김
    
    그래서 보통은:
    
    - 최근 **N턴만** messages로 유지하거나
    - 중간중간 **요약(summary)** 를 만들어 저장하고
        - `system: 요약` + `최근 대화` 형태로 컨텍스트를 구성해
    
B) 개인정보/보안
    
    - DB에는 암호화(또는 최소한 접근 제어), 로그 마스킹
    - API 키/토큰은 절대 대화 메시지에 저장하지 않기
    
C) 검색형 기능(RAG)까지 갈 경우
    
    - 전체 대화를 DB에 저장 + 중요한 메시지는 **벡터DB**에 임베딩 저장해서
        
        “예전에 했던 얘기”를 검색해서 다시 컨텍스트에 넣는 구조도 흔함

In [ ]:
# 스트리밍 방식 : 응답을 한 번에 받지 않고, 조각(chunk) 단위로 실시간 수신
stream = client.chat.completions.create(
    model='gpt-5.6-luna',
    messages= [{'role' : 'user', 'content' : '나 스트리밍 테스트 해 볼 건데, 아무말이나 길게 해줘봐'}],
    max_completion_tokens=2000,
    stream=True  # 스트리밍 모드 활성화 (chunk 단위로 변환)
)

for chunk in stream:  # stream에서 chunk 단위로 하나씩 순회하면서
    content = chunk.choices[0].delta.content  # 이번 chunk에서 새로 추가된 텍스트(delta)만 추출
    if content is not None:
        print(content, end='')  # 줄바꿈 없이 이어붙임

좋아요. 그럼 스트리밍 테스트에 도움이 되도록 문장 길이와 리듬을 조금씩 바꿔 가면서 자연스럽게 이어지는 이야기를 해볼게요. 지금부터는 특별한 목적 없이, 어느 평범한 하루에 갑자기 아주 작은 모험이 시작되었다는 설정입니다. 아침은 생각보다 조용했습니다. 창문 밖으로는 아직 완전히 깨어나지 않은 도시가 보였고, 멀리서 지나가는 자동차 소리와 누군가 현관문을 여닫는 소리가 희미하게 들렸습니다. 커튼 사이로 들어온 햇빛은 바닥에 길쭉한 모양을 만들고 있었고, 그 빛 위로 먼지가 천천히 떠다니고 있었습니다. 이상하게도 그런 장면을 보고 있으면 시간이 평소보다 느리게 흐르는 것처럼 느껴질 때가 있습니다.

식탁 위에는 어제 사 둔 빵 한 조각과 미지근한 물이 놓여 있었습니다. 냉장고를 열어 보니 먹을 만한 것은 많지 않았지만, 구석에서 작은 사과 하나를 발견했습니다. 사과를 씻으려고 물을 틀었는데, 물줄기가 생각보다 크게 들려서 잠깐 놀랐습니다. 별것 아닌 소리인데도 조용한 방에서는 꽤 선명하게 들립니다. 물이 흐르고, 접시가 살짝 부딪히고, 의자가 바닥을 긁는 소리까지 모두 하나의 작은 배경음처럼 이어집니다.

그날은 특별히 정해진 일정이 없었습니다. 그래서 오히려 어디든 가도 될 것 같았습니다. 가까운 공원에 갈 수도 있고, 한 번도 들어가 보지 않은 골목길을 걸어볼 수도 있고, 그냥 카페에 앉아서 지나가는 사람들을 바라볼 수도 있었습니다. 결국 신발을 신고 집을 나섰습니다. 계단을 내려가면서 몇 층에서 누군가 요리하는 냄새가 났고, 현관문을 열자 바깥의 공기가 생각보다 따뜻하게 느껴졌습니다. 하늘에는 구름이 조금 있었지만, 비가 올 것 같지는 않았습니다.

길을 걷다 보니 평소에는 보지 못했던 것들이 눈에 들어왔습니다. 담장 아래에 피어 있는 작은 노란 꽃, 오래된 간판 위에 앉아 있는 고양이, 자전거 바구니 안에 아무렇게나 들어 있는 장바구니, 그리고 신호등이 바뀌기를 기다리며 잠시 멈춰 선 사람들의 표정. 우리는 매일 같은 길을 지나가면서도 모든 것을 보지는

In [2]:
# 스트리밍 응답으로 실시간 출력되는 콘솔 챗봇
messages = [{'role': 'system', 'content': '당신은 지혜로운 챗봇입니다.'}]

print("종료하려면 exit를 입력하세요.")

while True:
    user_input = input('User : ')

    if user_input.lower().strip() == 'exit':
        print("대화 종료됩니다.")
        break

    print(f"User : {user_input}")
    messages.append({'role': 'user', 'content': user_input})

    response = client.chat.completions.create(
        model = 'gpt-5.6-luna',
        messages = messages,
        temperature = 1,
        top_p = 1,
        max_completion_tokens = 4096,  # 최대 출력 토큰 수 (응답 길이 제한)
        stream = True
    )
    
    print(f"Assistant : ", end='')
    assistant_message = ''
    for chunk in response:  # response에서 chunk 단위로 하나씩 순회하면서
        content = chunk.choices[0].delta.content  # 이번 chunk에서 새로 추가된 텍스트(delta)만 추출
        if content is not None:
            print(content, end='')  # 줄바꿈 없이 이어붙임
            assistant_message += content  # 답변 문자열에 누적
    print()

    # 완성된 답변은 히스토리에 저장해서 다음 턴 문맥 유지하는데 사용
    messages.append({'role': 'assistant', 'content': assistant_message})

종료하려면 exit를 입력하세요.
User : 오늘은 재미있는 일을 좀 하고싶어!
Assistant : 좋아! 오늘은 **“평소와 조금 다른 일”**을 해보자 😄

- **즉흥 미션:** 동네에서 처음 보는 카페에 가서 메뉴판 첫 번째 음료 주문하기
- **창작 놀이:** 10분 안에 짧은 소설이나 만화 만들기
- **탐험:** 지도에서 한 번도 안 가본 곳을 골라 산책하기
- **요리 실험:** 집에 있는 재료 3개로 새로운 간식 만들기
- **게임:** 내가 내는 수수께끼나 추리 문제 풀기
- **기분 전환:** 좋아하는 노래를 틀고 방 분위기 바꾸기
- **약간의 일탈:** 평소 안 입는 옷 조합으로 외출하기

원하면 지금 바로 **재미있는 퀴즈**, **추리 게임**, **즉흥 이야기 만들기**, **오늘의 랜덤 미션** 중 하나를 시작하자!
User : 재미있는 퀴즈
Assistant : 좋아! 첫 번째 퀴즈 🎉

**문제:**  
나는 입이 있지만 먹지 않고,  
침대가 있지만 자지 않아.  
나는 무엇일까?

① 책  
② 강  
③ 시계  
④ 집

정답 번호를 골라봐!
User : 4번 집
Assistant : 아쉽지만 정답은 **② 강**이야! 🌊

- **강의 입**: 강물이 바다로 흘러드는 곳을 ‘강어귀’ 또는 강의 입이라고 해.
- **강바닥**: 강이 흐르는 바닥을 말해.

집도 입과 침대가 있을 수 있지만, 이 수수께끼에서는 **강**이 정답이야 😄

다음 문제!

**문제 2:**  
아무리 많이 가져도 무게가 늘지 않는 것은 무엇일까?

① 나이  
② 물  
③ 돌  
④ 책
User : 나이
Assistant : 정답! 🎉 **① 나이**

나이는 아무리 많이 **먹어도** 몸무게가 늘지는 않지 😄

**문제 3:**  
세상에서 가장 뜨거운 과일은 무엇일까?

① 사과  
② 복숭아  
③ 천도복숭아  
④ 열대야 🍑
User : 천도복숭아
Assistant : 정답! 🎉 **③ 천도복숭아**

“천 도(1000도)”라서 가

KeyboardInterrupt: Interrupted by user